## PII Redactor Example Notebook


**Author**: Pooja Holkar ,
**email**:poholkar@in.ibm.com

Click link to open notebook in google colab:  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/data-prep-kit/data-prep-kit/blob/dev/examples/notebooks/PII/Run_your_first_PII_redactor_transform.ipynb)


### What is a PII Redactor?

A PII (Personally Identifiable Information) Redactor is a tool designed to identify and redact sensitive information in text data. PII includes details that can be used to identify an individual, such as:

Names
Email addresses
Phone numbers
Addresses
Financial details (e.g., credit card numbers)

### Overview of the use case
In this usecase, the PII Redactor is applied to text extracted from invoices to ensure sensitive customer information is not exposed during processing, sharing, or storage.

 **Workflow Overview**

- **Extracting and Converting Text:** The content of the invoice, originally in PDF format, is processed using the pdf2parquet transform to extract the text and convert it into a structured Parquet file, enabling easier handling and downstream processing.

- **Redacting Sensitive Information:** The generated Parquet file serves as the input for the pii_redactor_transform. This step scans the invoice data for personally identifiable information (PII) and applies masking techniques to redact any sensitive content, ensuring data privacy and compliance.

- **Creating the Final Output:** After the redaction process, a new output Parquet file is generated in **output-redacted** folder, containing the same structured data as the original but with all sensitive details securely masked to prevent unauthorized access or exposure.


### Why is PII Redaction Important?

 **Data Privacy Compliance**: Adheres to regulations like GDPR, HIPAA, or CCPA that mandate safeguarding customer information.

 **Risk Mitigation**: Prevents unauthorized access to or misuse of sensitive data.

 **Automation Benefits**: Simplifies and accelerates the process of securing information in large-scale document handling.


### Pre-req: Install data-prep-kit dependencies

In [1]:
%%capture
!pip install 'data-prep-toolkit-transforms[language]'
import pyarrow.parquet as pq
import pandas as pd

In [2]:
# Must enable nested asynchronous io in a notebook as the crawler uses coroutine to speed up acquisition and downloads
import nest_asyncio
nest_asyncio.apply()

import os

## Step 1: Configuration

### Download Data

In [3]:
import urllib.request
import shutil
shutil.os.makedirs("input-data", exist_ok=True)
urllib.request.urlretrieve("https://raw.githubusercontent.com/PoojaHolkar/data-prep-kit/refs/heads/dev/examples/notebooks/PII/input-data/Invoice.pdf")


('/var/folders/8f/xr0lnpcn30j8vvry3q_2bpww0000gn/T/tmpbf8bxoie',
 <http.client.HTTPMessage at 0x12713f450>)

In [4]:
# set input folder
input_folder = os.path.join("input-data")


#### We will place the downloaded files in the `input-data` folder. For our use case, we have used Invoice data file that will undergo processing. The output for each transform run will be generated in separate folders, with folder names following the format `files-<transform_name>`, making it easy to verify the respective transform outputs. This concludes the setup section.

## Step 2: Invoke Pdf2Parquet transform

In [5]:
%%capture
from dpk_pdf2parquet.transform_python import Pdf2Parquet
Pdf2Parquet(input_folder= input_folder,
               output_folder= "files-pdf2parquet",
               data_files_to_use=['.pdf'],
               pdf2parquet_contents_type='text/markdown').transform()

### Verify the input parquet created in output folder

## Step 3: Invoke PII Redactor configuration transform

In [6]:
%%capture
from dpk_pii_redactor.transform_python import PIIRedactor
PIIRedactor(input_folder='files-pdf2parquet',
            output_folder= 'files-piiredacted',
            pii_redactor_entities = ["PERSON", "EMAIL_ADDRESS","ORGANIZATION","PHONE_NUMBER", "LOCATION"],
            pii_redactor_operator = "replace",
            pii_redactor_transformed_contents = "title").transform()

## Step 4: Display Output in a Readable Format with masked PII information

In [7]:
data = pd.read_parquet('files-piiredacted/Invoice.parquet')
print(data["title"][0])
print(data["detected_pii"][0])


## INVOICE

<ORGANIZATION>.

Invoice Details:

Invoice Number: INV-2024-001

Invoice Date: November 15, 2024

Due Date: November 30, 2024

Billing Information:

Customer Name: <PERSON>

Address: 123 <LOCATION>, Apt 45, <LOCATION>, <LOCATION> 62704

Email: <EMAIL_ADDRESS>

Phone: <PHONE_NUMBER>

Shipping Information:

Recipient Name: <PERSON>

Address: 123 <LOCATION>, Apt 45, <LOCATION>, <LOCATION> 62704

## Item Details:

| Description               | Quantity   | Unit Price   | Total                               |
|---------------------------|------------|--------------|-------------------------------------|
| MacBook Air (13-inch, M2) | 1          | $999.00      | $999.00                             |
| 1                         |            | $199.00      | AppleCare+ for MacBook Air  $199.00 |

Payment Method: Credit Card (Visa)

Transaction ID: 9876543210ABCDE

Notes:

Thank you for your purchase!

For assistance, please contact our support team at <EMAIL_ADDRESS> or 1-800-MY-APP

<br>
<br>

### This notebook effectively demonstrates how to seamlessly apply redaction for PII entities